## Referencia histórica — no ejecutar como pipeline vigente

Este cuadernillo conserva la lógica original y su estado de ejecución histórico, que puede ser parcial o contener errores antiguos. Se preserva para consulta; no se han regenerado sus exportaciones retiradas.

Las limpiezas oficiales **03/05** y los notebooks de análisis **06/07** contienen esa lógica contrastada y sus resultados están ejecutados. Para revisar el flujo actual completo, consultar `Notebooks/README.md` y `GUIA_PASO_A_PASO_DATOS_Y_MODELOS.md` de la raíz. No interpretar resultados antiguos como resultados de la base vigente.

# 2 · Producción real y benchmark multi-anual

Este cuadernillo contiene el código completo de las dos fases: extracción de estimados y limpieza de producción real. No depende de importar un script para ocultar la lógica.

Regla de archivo faltante: si una semana no tiene archivo exacto, se usa el siguiente archivo disponible. Por ejemplo, `2026-S02` usa el archivo de `S03`; la decisión queda en `archivo_es_posterior=True` y `diferencia_semanas_archivo=1`.

Primero se puede ejecutar solamente la celda de producción. La celda final completa el benchmark cuando se quiera procesar también estimados.

In [1]:
"""Extraccion general de estimados semanales a cinco horizontes.

Este script es ejecutado por el notebook ``1. Extraccion de estimados.ipynb``.
Lee los archivos Excel de forma recursiva, extrae las filas TIPO=ESTIMADO de
la cache de tablas dinamicas y deja trazabilidad completa del archivo origen.

Caracteristicas importantes:
* no fija el procesamiento al ano 2026;
* si falta el archivo exacto de una semana, usa el siguiente disponible y
  conserva la diferencia en la trazabilidad;
* excluye por defecto archivos ajustados, pero los deja identificados en el
  inventario para que se puedan revisar;
* conserva los casos sin archivo, sin cache, sin campos requeridos o sin filas.
"""

from __future__ import annotations

from datetime import date, timedelta
from pathlib import Path
from zipfile import BadZipFile, ZipFile
import re
import unicodedata
import warnings
from concurrent.futures import ThreadPoolExecutor, as_completed

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

try:
    from lxml import etree
except ImportError as exc:  # pragma: no cover - mensaje util para el usuario
    raise ImportError(
        'Se necesita lxml para leer las caches de tablas dinamicas de Excel.'
    ) from exc


# -----------------------------------------------------------------------------
# Configuracion y rutas
# -----------------------------------------------------------------------------


def localizar_raiz_proyecto() -> Path:
    """Busca la raiz desde el directorio del kernel o sus padres."""

    for candidato in [Path.cwd(), *Path.cwd().parents]:
        if (candidato / 'Datos' / 'Estimados semanales').is_dir():
            return candidato

    raise FileNotFoundError(
        'No se encontro la carpeta Datos/Estimados semanales. '
        'Abra el notebook desde el proyecto o ajuste ROOT_PROYECTO.'
    )


ROOT_PROYECTO = localizar_raiz_proyecto()
RAIZ_ESTIMADOS = ROOT_PROYECTO / 'Datos' / 'Estimados semanales'
CARPETA_SALIDA = RAIZ_ESTIMADOS / 'datos extraidos'

NUMERO_HORIZONTES = 5
INCLUIR_AJUSTADOS = False
EXPORTAR_RESULTADOS = True
EXTENSIONES_EXCEL = {'.xlsx', '.xlsm', '.xlsb', '.xls'}

NS_EXCEL = 'http://schemas.openxmlformats.org/spreadsheetml/2006/main'


# -----------------------------------------------------------------------------
# Utilidades de normalizacion y lectura de nombres
# -----------------------------------------------------------------------------


def normalizar_texto(valor) -> str:
    if pd.isna(valor):
        return ''
    texto = str(valor).replace('\xa0', ' ').strip()
    texto = ''.join(
        caracter
        for caracter in unicodedata.normalize('NFKD', texto)
        if not unicodedata.combining(caracter)
    )
    return re.sub(r'\s+', ' ', texto).upper()


def convertir_entero(valor):
    try:
        if pd.isna(valor):
            return pd.NA
        numero = float(str(valor).replace(',', '.'))
        return int(numero) if numero.is_integer() else pd.NA
    except (TypeError, ValueError):
        return pd.NA


def ano_de_carpeta(ruta: Path):
    for parte in ruta.parts:
        if re.fullmatch(r'20\d{2}', parte):
            return int(parte)
    return pd.NA


def analizar_nombre_archivo(ruta: Path) -> dict:
    """Extrae ano y semana inicial sin asumir un patron unico de nombre."""

    nombre = ruta.stem
    nombre_normalizado = normalizar_texto(nombre)
    coincidencia_semana = re.search(
        r'\bSEM(?:ANA)?\s*[-_ ]*(\d{1,2})\b',
        nombre_normalizado,
        flags=re.IGNORECASE,
    )

    semana_inicio = int(coincidencia_semana.group(1)) if coincidencia_semana else pd.NA
    anos_en_nombre = [int(valor) for valor in re.findall(r'(?<!\d)(20\d{2})(?!\d)', nombre)]
    ano_inicio = anos_en_nombre[0] if anos_en_nombre else ano_de_carpeta(ruta)

    # Solo se usa como informacion auxiliar; la extraccion se basa en AÑO/SEMANA
    # almacenados en la cache, no en el rango escrito en el nombre.
    semana_fin = pd.NA
    if coincidencia_semana:
        resto = nombre_normalizado[coincidencia_semana.end():]
        siguiente = re.search(r'(?:-|A|HASTA)\s*(\d{1,2})(?!\d)', resto)
        if siguiente:
            valor_fin = int(siguiente.group(1))
            if 1 <= valor_fin <= 53:
                semana_fin = valor_fin

    es_ajustado = bool(re.search(r'\bAJUST(?:ADO|ADA)?\b|CORRECC|CORREG', nombre_normalizado))
    tiene_marca_envio = 'ENVIO' in nombre_normalizado
    tiene_sello_horario = bool(re.search(r'\d{8}[_ -]\d{6}', nombre))

    return {
        'archivo': ruta.name,
        'ruta_absoluta': str(ruta),
        'ruta_relativa': str(ruta.relative_to(RAIZ_ESTIMADOS)),
        'carpeta': ruta.parent.name,
        'ano_carpeta': ano_de_carpeta(ruta),
        'ano_inicio': ano_inicio,
        'semana_inicio': semana_inicio,
        'semana_fin_nombre': semana_fin,
        'es_ajustado': es_ajustado,
        'tiene_marca_envio': tiene_marca_envio,
        'tiene_sello_horario': tiene_sello_horario,
        'tamano_bytes': ruta.stat().st_size,
        'fecha_modificacion': pd.Timestamp(ruta.stat().st_mtime, unit='s'),
    }


def construir_inventario() -> pd.DataFrame:
    """Inventa todos los Excel de las subcarpetas de estimados."""

    filas = []
    for ruta in sorted(RAIZ_ESTIMADOS.rglob('*')):
        if not ruta.is_file() or ruta.suffix.lower() not in EXTENSIONES_EXCEL:
            continue
        if ruta.name.startswith('~$'):
            continue
        # Evita volver a leer archivos generados por este proceso si se usa
        # rglob sobre toda la raiz.
        if any(normalizar_texto(parte) == 'DATOS PREPROCESADOS' for parte in ruta.parts):
            continue
        if not re.search(r'ESTIM', normalizar_texto(ruta.name)):
            continue
        filas.append(analizar_nombre_archivo(ruta))

    inventario = pd.DataFrame(filas)
    if inventario.empty:
        return inventario

    inventario['semana_parseable'] = inventario['semana_inicio'].notna()
    inventario['ano_parseable'] = inventario['ano_inicio'].notna()
    inventario['candidato_extraccion'] = (
        inventario['semana_parseable']
        & inventario['ano_parseable']
        & (inventario['semana_inicio'].between(1, 53, inclusive='both'))
    )
    inventario['motivo_no_candidato'] = np.select(
        [
            ~inventario['semana_parseable'],
            ~inventario['ano_parseable'],
            ~inventario['semana_inicio'].between(1, 53, inclusive='both'),
        ],
        ['no se pudo leer la semana del nombre', 'no se pudo leer el ano', 'semana fuera de 1..53'],
        default='',
    )
    inventario = inventario.sort_values(
        ['ano_inicio', 'semana_inicio', 'ruta_relativa'],
        na_position='last',
    ).reset_index(drop=True)
    return inventario


def seleccionar_archivos(inventario: pd.DataFrame):
    """Selecciona un origen por ano/semana y devuelve duplicados aparte."""

    if inventario.empty:
        return inventario.copy(), inventario.copy(), inventario.copy()

    candidatos = inventario[inventario['candidato_extraccion']].copy()
    excluidos = inventario[~inventario['candidato_extraccion']].copy()
    if not INCLUIR_AJUSTADOS:
        excluidos = pd.concat(
            [excluidos, candidatos[candidatos['es_ajustado']]],
            ignore_index=True,
        )
        candidatos = candidatos[~candidatos['es_ajustado']].copy()

    # Se prioriza el archivo de envio, luego el que no tiene sello de descarga
    # duplicada, y finalmente el nombre mas corto. Los alternos no se borran:
    # quedan visibles en duplicados_archivos.
    candidatos['_prioridad'] = (
        candidatos['tiene_marca_envio'].astype(int) * -2
        + candidatos['tiene_sello_horario'].astype(int) * 2
        + candidatos['es_ajustado'].astype(int) * 10
    )
    candidatos = candidatos.sort_values(
        ['ano_inicio', 'semana_inicio', '_prioridad', 'archivo'],
        kind='stable',
    )
    seleccionados = candidatos.drop_duplicates(
        ['ano_inicio', 'semana_inicio'],
        keep='first',
    ).copy()
    alternos = candidatos.loc[~candidatos['ruta_absoluta'].isin(seleccionados['ruta_absoluta'])].copy()
    alternos['motivo'] = 'mas de un archivo para la misma ano/semana; no seleccionado'
    seleccionados = seleccionados.drop(columns=['_prioridad'], errors='ignore')
    alternos = alternos.drop(columns=['_prioridad'], errors='ignore')
    excluidos = excluidos.drop(columns=['_prioridad'], errors='ignore')
    return seleccionados, alternos, excluidos


def construir_solicitudes_semanales(seleccionados: pd.DataFrame) -> pd.DataFrame:
    """Asigna un archivo a cada semana, usando el siguiente si falta el exacto.

    Esta es la regla operativa del cuadernillo 1. Por ejemplo, si no existe
    archivo para 2026-S02 pero sí para 2026-S03, la solicitud S02 usa el
    archivo de S03. La semana solicitada y la semana del archivo se conservan
    separadas para que el benchmark pueda auditar la decisión.
    """

    if seleccionados.empty:
        return seleccionados.copy()

    solicitudes = []
    for ano, grupo in seleccionados.groupby('ano_inicio', dropna=True):
        ano = int(ano)
        grupo = grupo.sort_values('semana_inicio')
        semanas_disponibles = grupo['semana_inicio'].astype(int).tolist()
        for semana_solicitada in range(1, max(semanas_disponibles) + 1):
            candidatos = grupo.loc[grupo['semana_inicio'].ge(semana_solicitada)]
            if candidatos.empty:
                continue
            fila = candidatos.iloc[0].copy()
            semana_archivo = int(fila['semana_inicio'])
            fila['semana_solicitada'] = semana_solicitada
            fila['archivo_es_posterior'] = semana_archivo > semana_solicitada
            fila['diferencia_semanas_archivo'] = semana_archivo - semana_solicitada
            solicitudes.append(fila)

    return pd.DataFrame(solicitudes).reset_index(drop=True)


# -----------------------------------------------------------------------------
# Lectura de caches de tablas dinamicas
# -----------------------------------------------------------------------------


def valor_cache(elemento, valores_compartidos):
    etiqueta = etree.QName(elemento).localname
    if etiqueta == 'm':
        return None

    valor = elemento.get('v')
    if etiqueta == 'x':
        try:
            indice = int(valor)
            return valores_compartidos[indice] if indice < len(valores_compartidos) else None
        except (TypeError, ValueError):
            return None
    if etiqueta == 'n':
        try:
            numero = float(valor)
            return int(numero) if numero.is_integer() else numero
        except (TypeError, ValueError):
            return valor
    if etiqueta == 'b':
        return valor == '1'
    if valor is not None:
        return valor
    return ''.join(elemento.itertext()).strip() or None


def localizar_cache(libro: ZipFile):
    nombres = libro.namelist()
    definiciones = sorted(
        nombre for nombre in nombres
        if 'pivotcache/pivotcachedefinition' in nombre.lower()
        and '/_rels/' not in nombre.lower()
    )
    registros = sorted(
        nombre for nombre in nombres
        if 'pivotcache/pivotcacherecords' in nombre.lower()
    )
    if not definiciones or not registros:
        raise ValueError('el archivo no conserva una cache de tabla dinamica')
    return definiciones[0], registros[0]


def resolver_campos(columnas):
    aliases = {normalizar_texto(columna): columna for columna in columnas}
    requeridos = {}
    for canonico, opciones in {
        'TIPO': ['TIPO', 'TYPE'],
        'AÑO': ['AÑO', 'ANO', 'YEAR'],
        'SEMANA': ['SEMANA', 'WEEK'],
        'ESTI. EXPORTABLE': ['ESTI. EXPORTABLE', 'ESTI EXPORTABLE', 'TALLOS', 'ESTIMADO'],
    }.items():
        encontrado = next((aliases[normalizar_texto(opcion)] for opcion in opciones if normalizar_texto(opcion) in aliases), None)
        if encontrado is None:
            requeridos[canonico] = None
        else:
            requeridos[canonico] = encontrado
    faltantes = [campo for campo, original in requeridos.items() if original is None]
    if faltantes:
        raise ValueError(f'faltan campos requeridos: {faltantes}; disponibles: {columnas}')
    return requeridos


def extraer_desde_cache(ruta: Path, anos_interes: set[int]) -> pd.DataFrame:
    with ZipFile(ruta) as libro:
        definicion_nombre, registros_nombre = localizar_cache(libro)
        definicion = etree.fromstring(libro.read(definicion_nombre))
        campos_xml = definicion.find(f'{{{NS_EXCEL}}}cacheFields')
        if campos_xml is None:
            raise ValueError('la cache no tiene cacheFields')

        columnas = [campo.get('name') or f'campo_{i}' for i, campo in enumerate(campos_xml)]
        campos = resolver_campos(columnas)
        compartidos = []
        for campo in campos_xml:
            bloque = campo.find(f'{{{NS_EXCEL}}}sharedItems')
            compartidos.append(
                [] if bloque is None else [valor_cache(item, []) for item in bloque]
            )

        indice_tipo = columnas.index(campos['TIPO'])
        indice_ano = columnas.index(campos['AÑO'])
        filas = []

        with libro.open(registros_nombre) as flujo:
            contexto = etree.iterparse(
                flujo,
                events=('end',),
                tag=f'{{{NS_EXCEL}}}r',
            )
            for _, registro in contexto:
                celdas = list(registro)
                valores = [
                    valor_cache(celdas[posicion], compartidos[posicion])
                    if posicion < len(celdas) else None
                    for posicion in range(len(columnas))
                ]
                tipo = normalizar_texto(valores[indice_tipo])
                ano = convertir_entero(valores[indice_ano])
                if tipo == 'ESTIMADO' and pd.notna(ano) and int(ano) in anos_interes:
                    filas.append(dict(zip(columnas, valores)))

                registro.clear()
                padre = registro.getparent()
                while registro.getprevious() is not None:
                    del padre[0]

    resultado = pd.DataFrame(filas, columns=columnas)
    return resultado.rename(columns={original: canonico for canonico, original in campos.items()})


def extraer_desde_hojas(ruta: Path, anos_interes: set[int]) -> pd.DataFrame:
    """Fallback para archivos que no exponen cache; conserva un error visible."""

    if ruta.suffix.lower() == '.xlsb':
        engine = 'pyxlsb'
    elif ruta.suffix.lower() == '.xls':
        engine = 'xlrd'
    else:
        engine = None
    hojas = pd.read_excel(ruta, sheet_name=None, header=0, engine=engine)
    partes = []
    for hoja, tabla in hojas.items():
        columnas = {normalizar_texto(columna): columna for columna in tabla.columns}
        try:
            campos = resolver_campos(list(tabla.columns))
        except ValueError:
            continue
        tabla = tabla.rename(columns={original: canonico for canonico, original in campos.items()})
        tabla['TIPO'] = tabla['TIPO'].map(normalizar_texto)
        tabla['AÑO'] = tabla['AÑO'].map(convertir_entero)
        tabla = tabla[(tabla['TIPO'] == 'ESTIMADO') & tabla['AÑO'].isin(anos_interes)].copy()
        tabla['hoja_origen'] = hoja
        partes.append(tabla)
    if not partes:
        raise ValueError('no se encontro una hoja con los campos requeridos')
    return pd.concat(partes, ignore_index=True)


def leer_estimados_archivo(ruta: Path, anos_interes: set[int]):
    try:
        resultado = extraer_desde_cache(ruta, anos_interes)
        metodo = 'pivot_cache_xml'
    except (BadZipFile, KeyError, ValueError, OSError, ImportError) as error_cache:
        try:
            resultado = extraer_desde_hojas(ruta, anos_interes)
            metodo = 'hojas_excel_fallback'
        except Exception as error_hojas:
            raise RuntimeError(
                f'cache: {error_cache}; hojas: {type(error_hojas).__name__}: {error_hojas}'
            ) from error_hojas

    resultado = resultado.copy()
    resultado['AÑO'] = resultado['AÑO'].map(convertir_entero).astype('Int64')
    resultado['SEMANA'] = resultado['SEMANA'].map(convertir_entero).astype('Int64')
    return resultado, metodo


# -----------------------------------------------------------------------------
# Construccion de H1-H5 y controles
# -----------------------------------------------------------------------------


def construir_horizontes(ano_inicial: int, semana_inicial: int, numero_horizontes: int = 5):
    lunes_inicial = date.fromisocalendar(int(ano_inicial), int(semana_inicial), 1)
    filas = []
    for semanas_adelante in range(numero_horizontes):
        lunes_objetivo = lunes_inicial + timedelta(weeks=semanas_adelante)
        iso = lunes_objetivo.isocalendar()
        filas.append({
            'anio_objetivo': iso.year,
            'semana_objetivo': iso.week,
            'horizonte': semanas_adelante + 1,
            'semanas_adelante': semanas_adelante,
            'lunes_objetivo': pd.Timestamp(lunes_objetivo),
        })
    return pd.DataFrame(filas)


def preparar_estimado_h1_h5(base: pd.DataFrame, fila_archivo: pd.Series):
    semana_solicitada = int(fila_archivo.get('semana_solicitada', fila_archivo['semana_inicio']))
    horizontes = construir_horizontes(
        fila_archivo['ano_inicio'],
        semana_solicitada,
        NUMERO_HORIZONTES,
    )
    tabla = base.copy()
    tabla['AÑO'] = tabla['AÑO'].map(convertir_entero).astype('Int64')
    tabla['SEMANA'] = tabla['SEMANA'].map(convertir_entero).astype('Int64')
    resultado = tabla.merge(
        horizontes,
        left_on=['AÑO', 'SEMANA'],
        right_on=['anio_objetivo', 'semana_objetivo'],
        how='inner',
        validate='many_to_one',
    )
    resultado['anio_inicial_estimado'] = int(fila_archivo['ano_inicio'])
    resultado['semana_inicial_estimado'] = semana_solicitada
    resultado['lunes_inicial_estimado'] = pd.Timestamp(
        date.fromisocalendar(int(fila_archivo['ano_inicio']), semana_solicitada, 1)
    )
    resultado['archivo_origen'] = fila_archivo['archivo']
    resultado['ruta_relativa_origen'] = fila_archivo['ruta_relativa']
    resultado['carpeta_origen'] = fila_archivo['carpeta']
    resultado['metodo_lectura'] = fila_archivo.get('metodo_lectura', '')
    resultado['semana_archivo'] = int(fila_archivo['semana_inicio'])
    resultado['archivo_es_posterior'] = bool(fila_archivo.get('archivo_es_posterior', False))
    resultado['diferencia_semanas_archivo'] = int(
        fila_archivo.get('diferencia_semanas_archivo', 0)
    )
    return resultado


def construir_faltantes_archivos(
    seleccionados: pd.DataFrame,
    excluidos: pd.DataFrame | None = None,
    solicitudes: pd.DataFrame | None = None,
) -> pd.DataFrame:
    if seleccionados.empty:
        return pd.DataFrame()
    ajustados_por_clave = set()
    if excluidos is not None and not excluidos.empty:
        ajustados = excluidos[excluidos['es_ajustado'].eq(True)]
        ajustados_por_clave = set(
            zip(
                ajustados['ano_inicio'].dropna().astype(int),
                ajustados['semana_inicio'].dropna().astype(int),
            )
        )

    filas = []
    for ano, grupo in seleccionados.groupby('ano_inicio', dropna=True):
        ano = int(ano)
        max_semana = int(grupo['semana_inicio'].max())
        existentes = set(grupo['semana_inicio'].astype(int))
        for semana in range(1, max_semana + 1):
            es_ajustado_excluido = (ano, semana) in ajustados_por_clave
            filas.append({
                'ano': ano,
                'semana': semana,
                'esperada_hasta': max_semana,
                'archivo_disponible': semana in existentes,
                'tipo_faltante': (
                    'solo archivo ajustado excluido'
                    if semana not in existentes and es_ajustado_excluido
                    else 'sin archivo origen' if semana not in existentes else ''
                ),
                'archivo_ajustado_excluido': es_ajustado_excluido,
            })
    resultado = pd.DataFrame(filas)
    if solicitudes is not None and not solicitudes.empty and not resultado.empty:
        uso = solicitudes[[
            'ano_inicio', 'semana_solicitada', 'archivo', 'semana_inicio',
            'archivo_es_posterior', 'diferencia_semanas_archivo',
        ]].rename(columns={
            'ano_inicio': 'ano',
            'semana_solicitada': 'semana',
            'archivo': 'archivo_usado',
            'semana_inicio': 'semana_archivo_usada',
        })
        resultado = resultado.merge(uso, on=['ano', 'semana'], how='left')
        resultado['tipo_faltante'] = np.where(
            resultado['archivo_es_posterior'].eq(True),
            'se uso el siguiente archivo disponible',
            resultado['tipo_faltante'],
        )
    return resultado


def ejecutar_extraccion():
    inventario = construir_inventario()
    seleccionados, duplicados, excluidos = seleccionar_archivos(inventario)
    solicitudes = construir_solicitudes_semanales(seleccionados)

    def procesar_archivo(fila):
        ruta = Path(fila['ruta_absoluta'])
        ano = int(fila['ano_inicio'])
        semana = int(fila.get('semana_solicitada', fila['semana_inicio']))
        try:
            anos_interes = {
                construir_horizontes(ano, semana, NUMERO_HORIZONTES)['anio_objetivo'].min(),
                construir_horizontes(ano, semana, NUMERO_HORIZONTES)['anio_objetivo'].max(),
            }
            base, metodo = leer_estimados_archivo(ruta, anos_interes)
            fila = fila.copy()
            fila['metodo_lectura'] = metodo
            preparado = preparar_estimado_h1_h5(base, fila)
            control = {
                'ano_inicial': ano,
                'semana_inicial': semana,
                'semana_archivo': int(fila['semana_inicio']),
                'archivo_es_posterior': bool(fila.get('archivo_es_posterior', False)),
                'diferencia_semanas_archivo': int(fila.get('diferencia_semanas_archivo', 0)),
                'archivo': fila['archivo'],
                'ruta_relativa': fila['ruta_relativa'],
                'metodo_lectura': metodo,
                'filas_estimado_cache': len(base),
                'filas_h1_h5_extraidas': len(preparado),
                'estado': 'ok' if len(preparado) else 'sin filas H1-H5',
            }
            return preparado, control, None
        except Exception as error:
            error_detalle = {
                'ano_inicial': ano,
                'semana_inicial': semana,
                'semana_archivo': int(fila['semana_inicio']),
                'archivo': fila['archivo'],
                'ruta_relativa': fila['ruta_relativa'],
                'tipo_error': type(error).__name__,
                'detalle_error': str(error),
            }
            control = {
                'ano_inicial': ano,
                'semana_inicial': semana,
                'semana_archivo': int(fila['semana_inicio']),
                'archivo_es_posterior': bool(fila.get('archivo_es_posterior', False)),
                'diferencia_semanas_archivo': int(fila.get('diferencia_semanas_archivo', 0)),
                'archivo': fila['archivo'],
                'ruta_relativa': fila['ruta_relativa'],
                'metodo_lectura': '',
                'filas_estimado_cache': 0,
                'filas_h1_h5_extraidas': 0,
                'estado': 'error',
            }
            return pd.DataFrame(), control, error_detalle

    filas = [fila.copy() for _, fila in seleccionados.sort_values(
        ['ano_inicio', 'semana_inicio']
    ).iterrows()]
    partes = []
    errores = []
    controles_archivo = []
    total_archivos = len(filas)
    # Cada archivo contiene su propia cache XML. La lectura es independiente;
    # cuatro hilos reducen el tiempo sin cambiar la seleccion ni la logica.
    with ThreadPoolExecutor(max_workers=4) as executor:
        futuros = {
            executor.submit(procesar_archivo, fila): fila
            for fila in filas
        }
        for numero_archivo, futuro in enumerate(as_completed(futuros), start=1):
            fila = futuros[futuro]
            preparado, control, error = futuro.result()
            if not preparado.empty:
                partes.append(preparado)
            controles_archivo.append(control)
            if error is not None:
                errores.append(error)
            print(
                f'Procesado archivo {numero_archivo}/{total_archivos}: '
                f'{fila["ruta_relativa"]}',
                flush=True,
            )

    estimados = pd.concat(partes, ignore_index=True) if partes else pd.DataFrame()
    errores = pd.DataFrame(errores)
    controles_archivo = pd.DataFrame(controles_archivo)

    if estimados.empty:
        control_extraccion = pd.DataFrame(
            columns=['ano_inicial_estimado', 'semana_inicial_estimado', 'horizonte', 'registros', 'disponible']
        )
    else:
        claves = estimados[['anio_inicial_estimado', 'semana_inicial_estimado']].drop_duplicates()
        esperado = claves.merge(
            pd.DataFrame({'horizonte': range(1, NUMERO_HORIZONTES + 1)}),
            how='cross',
        )
        conteos = (
            estimados.groupby(
                ['anio_inicial_estimado', 'semana_inicial_estimado', 'horizonte'],
                as_index=False,
            ).size().rename(columns={'size': 'registros'})
        )
        control_extraccion = esperado.merge(
            conteos,
            on=['anio_inicial_estimado', 'semana_inicial_estimado', 'horizonte'],
            how='left',
        )
        control_extraccion['registros'] = control_extraccion['registros'].fillna(0).astype(int)
        control_extraccion['disponible'] = control_extraccion['registros'].gt(0)

    faltantes_horizontes = control_extraccion[~control_extraccion['disponible']].copy()
    if not faltantes_horizontes.empty:
        faltantes_horizontes['tipo_faltante'] = 'sin registros extraidos para el horizonte'

    faltantes_archivos = construir_faltantes_archivos(
        seleccionados,
        excluidos,
        solicitudes,
    )
    if not errores.empty:
        fallidos = errores[['ano_inicial', 'semana_inicial']].assign(
            archivo_disponible=True,
            tipo_faltante='archivo con error de lectura',
        )
        faltantes_archivos = faltantes_archivos.merge(
            fallidos,
            left_on=['ano', 'semana'],
            right_on=['ano_inicial', 'semana_inicial'],
            how='left',
            suffixes=('', '_error'),
        )
        faltantes_archivos['tipo_faltante'] = faltantes_archivos['tipo_faltante_error'].fillna(
            faltantes_archivos['tipo_faltante']
        )
        faltantes_archivos['archivo_disponible'] = faltantes_archivos['archivo_disponible_error'].fillna(
            faltantes_archivos['archivo_disponible']
        )
        faltantes_archivos = faltantes_archivos.drop(
            columns=['ano_inicial', 'semana_inicial', 'archivo_disponible_error', 'tipo_faltante_error'],
            errors='ignore',
        )

    resultados = {
        'inventario_archivos': inventario,
        'archivos_seleccionados': seleccionados,
        'solicitudes_semanales': solicitudes,
        'archivos_duplicados': duplicados,
        'archivos_excluidos': excluidos,
        'controles_archivo': controles_archivo,
        'errores_lectura': errores,
        'estimados_crudos_consolidados': estimados,
        'control_extraccion': control_extraccion,
        'faltantes_horizontes': faltantes_horizontes,
        'faltantes_archivos': faltantes_archivos,
    }
    return resultados


# -----------------------------------------------------------------------------
# EDA ligero y exportacion
# -----------------------------------------------------------------------------


def mostrar_tabla(tabla: pd.DataFrame, n: int = 30):
    try:
        display(tabla.head(n))
    except NameError:
        print(tabla.head(n).to_string(index=False))


def ejecutar_eda(resultados: dict):
    inventario = resultados['inventario_archivos']
    seleccionados = resultados['archivos_seleccionados']
    estimados = resultados['estimados_crudos_consolidados']
    faltantes_archivos = resultados['faltantes_archivos']
    faltantes_horizontes = resultados['faltantes_horizontes']
    errores = resultados['errores_lectura']

    print('Resumen de extraccion')
    print(f'Raiz de datos: {RAIZ_ESTIMADOS}')
    print(f'Archivos Excel identificados: {len(inventario):,}')
    print(f'Archivos seleccionados para extraer: {len(seleccionados):,}')
    print(f'Filas H1-H5 consolidadas: {len(estimados):,}')
    print(f'Archivos con error de lectura: {len(errores):,}')
    print(f'Filas de control faltantes: {len(faltantes_horizontes):,}')

    print('\nSemanas sin archivo dentro del rango observado por ano')
    mostrar_tabla(faltantes_archivos[faltantes_archivos['archivo_disponible'].eq(False)] if not faltantes_archivos.empty else faltantes_archivos)

    print('\nHorizontes sin registros extraidos')
    mostrar_tabla(faltantes_horizontes)

    print('\nArchivos alternos o duplicados no seleccionados')
    mostrar_tabla(resultados['archivos_duplicados'])

    print('\nArchivos ajustados excluidos por configuracion')
    mostrar_tabla(resultados['archivos_excluidos'][resultados['archivos_excluidos']['es_ajustado'].eq(True)] if not resultados['archivos_excluidos'].empty else resultados['archivos_excluidos'])

    print('\nErrores de lectura')
    mostrar_tabla(errores)

    if estimados.empty:
        return

    campos_clave = [
        'AÑO', 'SEMANA', 'TIPO', 'ESTI. EXPORTABLE',
        'anio_inicial_estimado', 'semana_inicial_estimado', 'horizonte',
    ]
    faltantes_campos = pd.DataFrame({
        'campo': [campo for campo in campos_clave if campo in estimados.columns],
        'nulos': [estimados[campo].isna().sum() for campo in campos_clave if campo in estimados.columns],
    })
    faltantes_campos['porcentaje'] = (faltantes_campos['nulos'] / len(estimados) * 100).round(2)
    print('\nNulos en campos clave')
    mostrar_tabla(faltantes_campos[faltantes_campos['nulos'].gt(0)])

    valores = pd.to_numeric(estimados['ESTI. EXPORTABLE'], errors='coerce')
    print('\nValores problematicos de ESTI. EXPORTABLE')
    print(f'No numericos: {valores.isna().sum():,}')
    print(f'Negativos: {(valores < 0).sum():,}')
    print(f'Ceros: {(valores == 0).sum():,}')

    duplicados = estimados.duplicated(
        subset=['anio_inicial_estimado', 'semana_inicial_estimado', 'horizonte', 'AÑO', 'SEMANA'],
        keep=False,
    )
    print(f'Filas repetidas en clave minima origen/objetivo: {duplicados.sum():,}')


def graficar_curvas(resultados: dict):
    estimados = resultados['estimados_crudos_consolidados']
    faltantes_archivos = resultados['faltantes_archivos']
    if estimados.empty:
        print('No hay datos para graficar.')
        return

    estimados = estimados.copy()
    estimados['tallos_estimados'] = pd.to_numeric(estimados['ESTI. EXPORTABLE'], errors='coerce')
    h1 = estimados[estimados['horizonte'].eq(1)].copy()
    curva = (
        h1.groupby(['anio_inicial_estimado', 'semana_inicial_estimado'], as_index=False)['tallos_estimados']
        .sum(min_count=1)
    )

    plt.figure(figsize=(14, 5))
    sns.lineplot(
        data=curva,
        x='semana_inicial_estimado',
        y='tallos_estimados',
        hue='anio_inicial_estimado',
        marker='o',
        palette='tab10',
    )
    plt.title('Curva H1 por ano y semana inicial')
    plt.xlabel('Semana ISO inicial')
    plt.ylabel('Tallos estimados')
    plt.xticks(range(1, 53, 2))
    plt.grid(alpha=0.25)
    plt.tight_layout()
    plt.show()

    cobertura = (
        estimados.groupby(['AÑO', 'SEMANA'], as_index=False)
        .size()
        .rename(columns={'size': 'registros'})
    )
    if not cobertura.empty:
        pivote = cobertura.pivot(index='AÑO', columns='SEMANA', values='registros')
        plt.figure(figsize=(16, max(2.5, 1.5 * len(pivote))))
        sns.heatmap(pivote, cmap='YlGnBu', annot=False, linewidths=0.2, cbar_kws={'label': 'Registros H1-H5'})
        plt.title('Cobertura de registros por ano/semana objetivo')
        plt.xlabel('Semana ISO objetivo')
        plt.ylabel('Ano ISO objetivo')
        plt.tight_layout()
        plt.show()


def exportar_resultados(resultados: dict):
    if not EXPORTAR_RESULTADOS:
        return
    CARPETA_SALIDA.mkdir(parents=True, exist_ok=True)
    nombres = {
        'inventario_archivos': 'inventario_archivos.csv',
        'archivos_seleccionados': 'archivos_seleccionados.csv',
        'solicitudes_semanales': 'solicitudes_semanales.csv',
        'archivos_duplicados': 'archivos_duplicados.csv',
        'archivos_excluidos': 'archivos_excluidos.csv',
        'controles_archivo': 'control_por_archivo.csv',
        'errores_lectura': 'errores_lectura.csv',
        'control_extraccion': 'control_extraccion_h1_h5.csv',
        'faltantes_horizontes': 'faltantes_horizontes.csv',
        'faltantes_archivos': 'faltantes_archivos.csv',
        'estimados_crudos_consolidados': 'estimados_crudos_h1_h5.csv',
    }
    for clave, nombre in nombres.items():
        tabla = resultados.get(clave)
        if isinstance(tabla, pd.DataFrame):
            tabla.to_csv(CARPETA_SALIDA / nombre, index=False, encoding='utf-8-sig')

    estimados = resultados['estimados_crudos_consolidados']
    if not estimados.empty:
        estimados[estimados['horizonte'].eq(1)].to_csv(
            CARPETA_SALIDA / 'estimados_H1.csv',
            index=False,
            encoding='utf-8-sig',
        )
    print(f'\nResultados exportados en: {CARPETA_SALIDA}')




In [2]:
"""Construye las bases limpias de produccion y el benchmark para todos los anos.

La logica de limpieza sigue el notebook 03:

* se leen las hojas de detalle de cada libro anual;
* se eliminan las columnas operativas que no entran al benchmark;
* se eliminan filas completamente vacias y se reinicia el indice;
* ``Tipo Corte`` se imputa solo cuando existe una moda en la semana anterior,
  actual o siguiente para la misma combinacion producto-color-variedad;
* las llaves del benchmark se normalizan y la produccion se agrega por semana;
* se intenta primero la union estricta y luego una union por variedad para los
  registros que no coincidieron por color.

El proceso no convierte faltantes de estimado o produccion en cero. Esa
decision queda para la capa de metricas, donde tambien se conserva la marca de
que el volumen proviene de una fuente ausente.
"""

from __future__ import annotations

from datetime import time
from importlib.util import module_from_spec, spec_from_file_location
from pathlib import Path
import json
import re
import unicodedata
import warnings

import numpy as np
import pandas as pd


ROOT = next(ruta for ruta in [Path.cwd(), *Path.cwd().parents] if (ruta / 'Datos').is_dir() and (ruta / 'Notebooks').is_dir())
DATOS = ROOT / 'Datos'
ANOS = (2021, 2022, 2023, 2024, 2025, 2026)
HOJAS_DETALLE = {
    2021: ('Sheet1',),
    2022: ('Sheet1',),
    2023: ('Sheet1', 'Hoja1'),
    2024: ('Sheet1', 'sem 45 en adelante'),
    2025: ('01-44', '45-52'),
    2026: ('BD',),
}

COLUMNAS_FINALES_LIMPIAS = [
    'Finca', 'Invernadero', 'Fecha Corte', 'Año', 'Semana', 'Tipo Corte',
    'Producto', 'Grado', 'Color', 'Variedad', 'Tallos', 'Lote',
    'Estado Inventario', 'Hora Corte', 'Tipo Inventario',
]
COLUMNAS_CLAVE_VACIAS = [
    'Finca', 'Invernadero', 'Fecha Corte', 'Año', 'Semana', 'Tipo Corte',
    'Producto', 'Grado', 'Color', 'Variedad', 'Tallos', 'Lote',
    'Estado Inventario', 'Hora Corte', 'Tipo Inventario',
]
COLUMNAS_LLAVE = ['anio', 'semana', 'k_finca', 'k_producto', 'k_color', 'k_variedad']


def normalizar_texto(valor) -> str:
    if pd.isna(valor):
        return ''
    texto = str(valor).replace('\xa0', ' ').strip()
    texto = unicodedata.normalize('NFKD', texto)
    texto = ''.join(caracter for caracter in texto if not unicodedata.combining(caracter))
    return re.sub(r'\s+', ' ', texto).upper()


def normalizar_columna(valor) -> str:
    return re.sub(r'[^A-Z0-9]+', ' ', normalizar_texto(valor)).strip()


DIR_PRODUCCION = next(
    (ruta for ruta in DATOS.iterdir() if ruta.is_dir() and 'PRODUCCION' in normalizar_texto(ruta.name)),
    DATOS / 'Produccion real',
)
# Salida independiente del antiguo esquema de preprocesados.
SALIDA = DIR_PRODUCCION / 'datos_limpios_multianual'


def resolver_columna(columnas, nombre: str):
    objetivo = normalizar_columna(nombre)
    for columna in columnas:
        if normalizar_columna(columna) == objetivo:
            return columna
    return None


def archivo_produccion(anio: int) -> Path:
    candidatos = sorted(
        ruta for ruta in DIR_PRODUCCION.glob('*.xlsx')
        if str(anio) in ruta.stem and 'ACUMULADA' in normalizar_texto(ruta.stem)
    )
    if not candidatos:
        raise FileNotFoundError(f'No se encontro el libro de produccion para {anio}.')
    return candidatos[0]


def _leer_hoja_detalle(ruta: Path, hoja: str, anio: int) -> pd.DataFrame:
    encabezado = pd.read_excel(ruta, sheet_name=hoja, header=0, nrows=0, engine='calamine')
    requeridas = set(normalizar_columna(columna) for columna in COLUMNAS_FINALES_LIMPIAS)
    usecols = [
        columna for columna in encabezado.columns
        if normalizar_columna(columna) in requeridas
    ]
    if not usecols:
        raise ValueError(f'La hoja {hoja!r} no tiene columnas de detalle reconocibles.')

    tabla = pd.read_excel(
        ruta,
        sheet_name=hoja,
        header=0,
        usecols=usecols,
        engine='calamine',
    )
    renombrar = {}
    for columna in tabla.columns:
        canonica = next(
            (nombre for nombre in COLUMNAS_FINALES_LIMPIAS
             if normalizar_columna(nombre) == normalizar_columna(columna)),
            None,
        )
        if canonica is not None:
            renombrar[columna] = canonica
    tabla = tabla.rename(columns=renombrar)
    for columna in COLUMNAS_FINALES_LIMPIAS:
        if columna not in tabla.columns:
            tabla[columna] = pd.NA
    tabla = tabla[COLUMNAS_FINALES_LIMPIAS].copy()
    tabla['archivo_origen'] = ruta.name
    tabla['hoja_origen'] = hoja
    tabla['ano_libro_origen'] = anio
    return tabla


def _hora(valor):
    if pd.isna(valor):
        return None
    if isinstance(valor, time):
        return valor
    if isinstance(valor, pd.Timestamp):
        return valor.time()
    if isinstance(valor, (int, float, np.integer, np.floating)) and 0 <= float(valor) < 1:
        segundos = int(round(float(valor) * 24 * 60 * 60)) % (24 * 60 * 60)
        return time(second=segundos % 60, minute=(segundos // 60) % 60, hour=segundos // 3600)
    convertido = pd.to_datetime(str(valor).strip(), errors='coerce')
    return None if pd.isna(convertido) else convertido.time()


def _imputar_tipo_corte(tabla: pd.DataFrame):
    antes = int(tabla['Tipo Corte'].isna().sum())
    tabla['Tipo Corte'] = (
        tabla['Tipo Corte'].astype('string').str.strip().replace({'': pd.NA, '<NA>': pd.NA})
    )
    if antes == 0:
        return tabla, 0

    tabla['_semana_lunes'] = tabla['Fecha Corte'].dt.to_period('W-SUN').dt.start_time
    conteo = (
        tabla.loc[
            tabla['Tipo Corte'].notna()
            & tabla['Producto'].notna()
            & tabla['Color'].notna()
            & tabla['Variedad'].notna()
            & tabla['_semana_lunes'].notna()
        ]
        .groupby(['Producto', 'Color', 'Variedad', '_semana_lunes', 'Tipo Corte'], dropna=False)
        .size()
        .reset_index(name='Cantidad')
    )
    if conteo.empty:
        tabla.drop(columns='_semana_lunes', inplace=True)
        return tabla, 0

    candidatos = pd.concat([
        conteo.assign(SemanaObjetivo=conteo['_semana_lunes'] - pd.Timedelta(days=7)),
        conteo.assign(SemanaObjetivo=conteo['_semana_lunes']),
        conteo.assign(SemanaObjetivo=conteo['_semana_lunes'] + pd.Timedelta(days=7)),
    ], ignore_index=True)
    candidatos = (
        candidatos
        .groupby(['Producto', 'Color', 'Variedad', 'SemanaObjetivo', 'Tipo Corte'], dropna=False)['Cantidad']
        .sum()
        .reset_index()
        .sort_values(['Producto', 'Color', 'Variedad', 'SemanaObjetivo', 'Cantidad', 'Tipo Corte'],
                     ascending=[True, True, True, True, False, True], kind='stable')
        .drop_duplicates(['Producto', 'Color', 'Variedad', 'SemanaObjetivo'])
    )
    moda = candidatos.set_index(
        ['Producto', 'Color', 'Variedad', 'SemanaObjetivo']
    )['Tipo Corte']
    faltantes = tabla['Tipo Corte'].isna()
    claves = pd.MultiIndex.from_frame(
        tabla.loc[faltantes, ['Producto', 'Color', 'Variedad', '_semana_lunes']]
        .rename(columns={'_semana_lunes': 'SemanaObjetivo'})
    )
    imputados = moda.reindex(claves).to_numpy()
    tabla.loc[faltantes, 'Tipo Corte'] = imputados
    tabla.drop(columns='_semana_lunes', inplace=True)
    despues = int(tabla['Tipo Corte'].isna().sum())
    return tabla, antes - despues


def limpiar_produccion(tabla: pd.DataFrame):
    tabla = tabla.copy()
    filas_iniciales = len(tabla)
    tabla['Fecha Corte'] = pd.to_datetime(tabla['Fecha Corte'], errors='coerce')
    filas_vacias = tabla[COLUMNAS_CLAVE_VACIAS].isna().all(axis=1)
    tabla = tabla.loc[~filas_vacias].reset_index(drop=True)
    tabla, cantidad_imputada = _imputar_tipo_corte(tabla)

    for columna in ['Año', 'Semana', 'Tallos', 'Lote']:
        tabla[columna] = pd.to_numeric(tabla[columna], errors='coerce').astype('Int64')
    tabla['Hora Corte'] = tabla['Hora Corte'].map(_hora)

    iso = tabla['Fecha Corte'].dt.isocalendar()
    tabla['anio_iso'] = iso.year.astype('Int64')
    tabla['semana_iso'] = iso.week.astype('Int64')
    tabla.attrs['control'] = {
        'filas_iniciales': filas_iniciales,
        'filas_completamente_vacias': int(filas_vacias.sum()),
        'filas_finales': len(tabla),
        'tipo_corte_imputados': cantidad_imputada,
        'tipo_corte_faltantes': int(tabla['Tipo Corte'].isna().sum()),
        'fechas_invalidas': int(tabla['Fecha Corte'].isna().sum()),
    }
    return tabla


def limpiar_nombre(valor):
    nombre = normalizar_texto(valor)
    return nombre if nombre else pd.NA


def limpiar_finca(valor):
    if pd.notna(valor) and str(valor).strip() == '¡':
        return 'GAITANA'
    nombre = limpiar_nombre(valor)
    mapa = {
        'GF': 'GAITANA', 'LA GAITANA': 'GAITANA', 'GAITANA': 'GAITANA',
        'AR': 'ARABELLA', 'ARABELLA': 'ARABELLA',
        'CABANA': 'CABANA', 'LA CABANA': 'CABANA', 'FINCA LA CABANA': 'CABANA',
    }
    return mapa.get(nombre, nombre) if pd.notna(nombre) else pd.NA


def agregar_produccion(tabla: pd.DataFrame):
    prod = tabla.copy()
    prod['fecha'] = prod['Fecha Corte']
    prod['anio'] = prod['anio_iso']
    prod['semana'] = prod['semana_iso']
    prod['k_finca'] = prod['Finca'].map(limpiar_finca)
    prod['k_producto'] = prod['Producto'].map(limpiar_nombre)
    prod['k_color'] = prod['Color'].map(limpiar_nombre)
    prod['k_variedad'] = prod['Variedad'].map(limpiar_nombre)
    prod['tallos_reales'] = pd.to_numeric(prod['Tallos'], errors='coerce')
    cabana = prod['k_finca'].eq('CABANA').fillna(False)
    validas = prod.loc[
        ~cabana & prod[COLUMNAS_LLAVE].notna().all(axis=1)
    ].copy()
    agregado = validas.groupby(COLUMNAS_LLAVE, as_index=False, dropna=False).agg(
        tallos_reales=('tallos_reales', 'sum')
    )
    return agregado, prod, cabana


def cargar_y_limpiar_produccion():
    agregados = []
    controles = []
    controles_semanas = []
    controles_cabana = []
    rutas_salida = []

    for anio in ANOS:
        ruta = archivo_produccion(anio)
        partes = []
        hojas_ok = []
        errores_hojas = []
        for hoja in HOJAS_DETALLE[anio]:
            try:
                partes.append(_leer_hoja_detalle(ruta, hoja, anio))
                hojas_ok.append(hoja)
            except Exception as error:
                errores_hojas.append(f'{hoja}: {type(error).__name__}: {error}')
        if not partes:
            raise RuntimeError(f'No se pudo leer ninguna hoja de detalle de {ruta.name}: {errores_hojas}')

        limpio = limpiar_produccion(pd.concat(partes, ignore_index=True))
        # Entre libros, algunas columnas categóricas mezclan texto, booleanos
        # y números. StringDtype conserva los faltantes y permite almacenar
        # todos los años en Parquet con un esquema estable.
        columnas_objeto = [columna for columna in limpio.columns if limpio[columna].dtype == object]
        for columna in columnas_objeto:
            if columna != 'Hora Corte':
                limpio[columna] = limpio[columna].astype('string')
        salida_anual = SALIDA / 'produccion_limpia_por_anio' / f'produccion_limpia_{anio}.parquet'
        salida_anual.parent.mkdir(parents=True, exist_ok=True)
        limpio.to_parquet(salida_anual, index=False, compression='snappy')
        rutas_salida.append(str(salida_anual))

        agregado, prod, cabana = agregar_produccion(limpio)
        agregados.append(agregado)
        controles_cabana.append({
            'anio': anio,
            'filas_excluidas_cabana': int(cabana.sum()),
            'tallos_excluidos_cabana': prod.loc[cabana, 'tallos_reales'].sum(min_count=1),
        })
        grupo = prod.loc[~cabana & prod['anio'].notna() & prod['semana'].notna()].copy()
        for (anio_iso, semana), bloque in grupo.groupby(['anio', 'semana'], dropna=False):
            lunes = pd.Timestamp.fromisocalendar(int(anio_iso), int(semana), 1)
            fechas = set(bloque['fecha'].dropna().dt.normalize())
            faltantes = [lunes + pd.Timedelta(days=i) for i in range(6) if lunes + pd.Timedelta(days=i) not in fechas]
            controles_semanas.append({
                'anio': int(anio_iso), 'semana': int(semana),
                'dias_con_produccion': len(fechas),
                'dias_faltantes_lunes_sabado': len(faltantes),
                'fechas_faltantes': ', '.join(x.strftime('%Y-%m-%d') for x in sorted(faltantes)),
                'estado_semana': 'completa' if not faltantes else 'revisar_festivo_o_faltante',
            })

        control = dict(limpio.attrs.get('control', {}))
        control.update({
            'ano_libro': anio,
            'archivo_origen': ruta.name,
            'hojas_leidas': ', '.join(hojas_ok),
            'errores_hojas': ' | '.join(errores_hojas),
            'min_fecha': limpio['Fecha Corte'].min(),
            'max_fecha': limpio['Fecha Corte'].max(),
            'filas_llave_invalida_benchmark': int((
                limpio['anio_iso'].isna()
                | limpio['semana_iso'].isna()
                | limpio['Finca'].map(limpiar_finca).isna()
                | limpio['Producto'].map(limpiar_nombre).isna()
                | limpio['Color'].map(limpiar_nombre).isna()
                | limpio['Variedad'].map(limpiar_nombre).isna()
            ).sum()),
            'salida_parquet': str(salida_anual),
        })
        controles.append(control)
        print(f'Produccion {anio}: {len(limpio):,} filas limpias -> {salida_anual.name}')

    produccion_agregada = pd.concat(agregados, ignore_index=True)
    produccion_agregada = (
        produccion_agregada.groupby(COLUMNAS_LLAVE, as_index=False, dropna=False)
        .agg(tallos_reales=('tallos_reales', 'sum'))
    )
    return (
        produccion_agregada,
        pd.DataFrame(controles),
        pd.DataFrame(controles_semanas),
        pd.DataFrame(controles_cabana),
        rutas_salida,
    )


def cargar_extraccion_estimados():
    # La extracción está definida en la primera celda; no depende de scripts externos.
    resultados = ejecutar_extraccion()
    exportar_resultados(resultados)
    return resultados


def preparar_estimados(estimados: pd.DataFrame):
    if estimados.empty:
        return pd.DataFrame(columns=COLUMNAS_LLAVE + ['tallos_estimados']), pd.DataFrame()
    est = estimados.copy()
    if 'horizonte' in est.columns:
        est = est.loc[pd.to_numeric(est['horizonte'], errors='coerce').eq(1)].copy()

    def col(nombre, alternativas=()):
        for opcion in (nombre, *alternativas):
            encontrada = resolver_columna(est.columns, opcion)
            if encontrada is not None:
                return encontrada
        return None

    c_anio = col('Año', ('ANO', 'YEAR'))
    c_semana = col('Semana', ('WEEK',))
    c_finca = col('CODFINCA', ('FINCA', 'FARM'))
    c_producto = col('Producto', ('PRODUCTO',))
    c_color = col('Color', ('COLOR',))
    c_variedad = col('Variedad', ('VARIEDAD',))
    c_tallos = col('ESTI. EXPORTABLE', ('ESTI EXPORTABLE', 'TALLOS', 'ESTIMADO'))
    faltantes = [nombre for nombre, valor in {
        'Año': c_anio, 'Semana': c_semana, 'CODFINCA': c_finca,
        'Producto': c_producto, 'Color': c_color, 'Variedad': c_variedad,
        'ESTI. EXPORTABLE': c_tallos,
    }.items() if valor is None]
    if faltantes:
        raise ValueError(f'Faltan columnas en estimados H1: {faltantes}')

    est['anio'] = pd.to_numeric(est[c_anio], errors='coerce').astype('Int64')
    est['semana'] = pd.to_numeric(est[c_semana], errors='coerce').astype('Int64')
    est['k_finca'] = est[c_finca].map(limpiar_finca)
    est['k_producto'] = est[c_producto].map(limpiar_nombre)
    est['k_color'] = est[c_color].map(limpiar_nombre)
    est['k_variedad'] = est[c_variedad].map(limpiar_nombre)
    est['tallos_estimados'] = pd.to_numeric(est[c_tallos], errors='coerce')
    est = est.loc[est[COLUMNAS_LLAVE].notna().all(axis=1)].copy()
    return est, pd.DataFrame({
        'filas_h1_originales': [len(estimados)],
        'filas_h1_llave_valida': [len(est)],
        'filas_h1_llave_invalida': [len(estimados) - len(est)],
    })


def unir_benchmark(estimados: pd.DataFrame, produccion: pd.DataFrame):
    estimado = estimados.copy()
    produccion = produccion.copy()
    cabana_est = estimado['k_finca'].eq('CABANA').fillna(False)
    cabana_prod = produccion['k_finca'].eq('CABANA').fillna(False)
    control_cabana = pd.DataFrame({
        'fuente': ['estimado', 'produccion'],
        'filas_excluidas': [int(cabana_est.sum()), int(cabana_prod.sum())],
        'tallos_excluidos': [
            estimado.loc[cabana_est, 'tallos_estimados'].sum(min_count=1),
            produccion.loc[cabana_prod, 'tallos_reales'].sum(min_count=1),
        ],
    })
    estimado = estimado.loc[~cabana_est].copy()
    produccion = produccion.loc[~cabana_prod].copy()

    ultima_fecha = produccion['fecha'].max() if 'fecha' in produccion else pd.NaT
    if pd.isna(ultima_fecha):
        raise ValueError('No hay fecha valida de produccion para definir el corte.')
    corte = ultima_fecha.normalize()
    lunes_corte = pd.Timestamp.fromisocalendar(corte.isocalendar().year, corte.isocalendar().week, 1)
    estimado['lunes_semana'] = pd.to_datetime(
        estimado['anio'].astype(str) + '-W' + estimado['semana'].astype(str).str.zfill(2) + '-1',
        format='%G-W%V-%u', errors='coerce'
    )
    fuera_corte = estimado['lunes_semana'] > lunes_corte
    invalidas_fecha = estimado['lunes_semana'].isna()
    control_excluidos = pd.DataFrame([
        {'motivo': f'Posterior a {corte.date()}', 'filas': int(fuera_corte.sum()),
         'tallos': estimado.loc[fuera_corte, 'tallos_estimados'].sum(min_count=1)},
        {'motivo': 'Año o semana invalidos', 'filas': int(invalidas_fecha.sum()),
         'tallos': estimado.loc[invalidas_fecha, 'tallos_estimados'].sum(min_count=1)},
    ])
    estimado = estimado.loc[~fuera_corte & ~invalidas_fecha].copy()

    est_agregado = estimado.groupby(COLUMNAS_LLAVE, as_index=False, dropna=False).agg(
        tallos_estimados=('tallos_estimados', 'sum')
    )
    prod_agregado = produccion.groupby(COLUMNAS_LLAVE, as_index=False, dropna=False).agg(
        tallos_reales=('tallos_reales', 'sum')
    )
    union = est_agregado.merge(prod_agregado, on=COLUMNAS_LLAVE, how='outer', indicator=True, validate='one_to_one')
    union['estado'] = union['_merge'].map({
        'both': 'coincide', 'left_only': 'solo_estimado', 'right_only': 'solo_produccion'
    })
    union = union.drop(columns='_merge')
    coincidencias = union.loc[union['estado'].eq('coincide')].copy()
    est_pendiente = union.loc[union['estado'].eq('solo_estimado')].copy()
    prod_pendiente = union.loc[union['estado'].eq('solo_produccion')].copy()

    llave_variedad = ['anio', 'semana', 'k_finca', 'k_producto', 'k_variedad']
    if not est_pendiente.empty or not prod_pendiente.empty:
        est_var = est_pendiente.groupby(llave_variedad, as_index=False, dropna=False).agg(
            k_color=('k_color', 'first'), tallos_estimados=('tallos_estimados', 'sum')
        )
        prod_var = prod_pendiente.groupby(llave_variedad, as_index=False, dropna=False).agg(
            k_color=('k_color', 'first'), tallos_reales=('tallos_reales', 'sum')
        )
        por_variedad = est_var.merge(prod_var, on=llave_variedad, how='outer', indicator=True, suffixes=('_est', '_prod'))
        por_variedad['k_color'] = por_variedad['k_color_est'].combine_first(por_variedad['k_color_prod'])
        # Cada tabla ya conserva una sola medida despues del groupby:
        # est_var aporta tallos_estimados y prod_var aporta tallos_reales.
        # No se crean columnas con sufijo para estas dos medidas.
        por_variedad['tallos_estimados'] = por_variedad['tallos_estimados']
        por_variedad['tallos_reales'] = por_variedad['tallos_reales']
        por_variedad['estado'] = por_variedad['_merge'].map({
            'both': 'unido_variedad', 'left_only': 'solo_estimado', 'right_only': 'solo_produccion'
        })
        recuperados = por_variedad[COLUMNAS_LLAVE + ['tallos_estimados', 'tallos_reales', 'estado']].copy()
    else:
        recuperados = pd.DataFrame(columns=COLUMNAS_LLAVE + ['tallos_estimados', 'tallos_reales', 'estado'])
    final = pd.concat([coincidencias, recuperados], ignore_index=True)
    final = final[COLUMNAS_LLAVE + ['tallos_estimados', 'tallos_reales', 'estado']]
    final = final.sort_values(COLUMNAS_LLAVE).reset_index(drop=True)
    if final.duplicated(COLUMNAS_LLAVE).any():
        raise ValueError('La union final produjo llaves duplicadas.')
    return final, control_cabana, control_excluidos, {
        'coincidencias_estrictas': len(coincidencias),
        'solo_estimado_estricta': len(est_pendiente),
        'solo_produccion_estricta': len(prod_pendiente),
        'recuperadas_por_variedad': int((recuperados['estado'] == 'unido_variedad').sum()),
    }


def exportar(resultados):
    SALIDA.mkdir(parents=True, exist_ok=True)
    for nombre, tabla in resultados['tablas'].items():
        if isinstance(tabla, pd.DataFrame):
            tabla.to_csv(SALIDA / f'{nombre}.csv', index=False, encoding='utf-8-sig')
    resultados['benchmark'].to_parquet(SALIDA / 'benchmark_consolidado_multianual.parquet', index=False)
    resultados['benchmark'].to_excel(SALIDA / 'benchmark_consolidado_multianual.xlsx', index=False)
    (SALIDA / 'manifest.json').write_text(
        json.dumps({
            'anos': ANOS,
            'salida': str(SALIDA),
            'filas_benchmark': len(resultados['benchmark']),
            'fecha_maxima_produccion': str(resultados['fecha_maxima_produccion']),
            'archivos_produccion_limpia': resultados['archivos_produccion_limpia'],
        }, ensure_ascii=False, indent=2),
        encoding='utf-8',
    )


def ejecutar_solo_produccion():
    """Ejecuta únicamente la importación y limpieza de producción real."""

    (
        produccion,
        control_prod,
        control_semanas,
        control_cabana,
        archivos_limpios,
    ) = cargar_y_limpiar_produccion()
    SALIDA.mkdir(parents=True, exist_ok=True)
    control_prod.to_csv(SALIDA / 'control_produccion.csv', index=False, encoding='utf-8-sig')
    control_semanas.to_csv(SALIDA / 'control_semanas_produccion.csv', index=False, encoding='utf-8-sig')
    control_cabana.to_csv(SALIDA / 'control_cabana_produccion.csv', index=False, encoding='utf-8-sig')
    produccion.to_csv(
        SALIDA / 'produccion_semanal_limpia_multianual.csv',
        index=False,
        encoding='utf-8-sig',
    )
    produccion.to_parquet(
        SALIDA / 'produccion_semanal_limpia_multianual.parquet',
        index=False,
        compression='snappy',
    )
    fecha_maxima = max(
        pd.to_datetime(control_prod['max_fecha'], errors='coerce').dropna(),
        default=pd.NaT,
    )
    resultado = {
        'produccion': produccion,
        'control_produccion': control_prod,
        'control_semanas': control_semanas,
        'control_cabana': control_cabana,
        'archivos_produccion_limpia': archivos_limpios,
        'fecha_maxima_produccion': fecha_maxima,
    }
    print(f'Produccion real limpia: {len(produccion):,} filas semanales agregadas')
    print(f'Salida: {SALIDA}')
    return resultado


def ejecutar():
    fase_produccion = ejecutar_solo_produccion()
    produccion = fase_produccion['produccion']
    control_prod = fase_produccion['control_produccion']
    control_semanas = fase_produccion['control_semanas']
    control_cabana_prod = fase_produccion['control_cabana']
    archivos_limpios = fase_produccion['archivos_produccion_limpia']
    extraccion = cargar_extraccion_estimados()
    estimados, control_est = preparar_estimados(extraccion['estimados_crudos_consolidados'])
    # Se conserva fecha para el corte global; no se mezcla con el detalle crudo.
    prod_detalle = []
    for archivo in archivos_limpios:
        parte = pd.read_parquet(archivo, columns=['Fecha Corte', 'anio_iso', 'semana_iso', 'Finca', 'Producto', 'Color', 'Variedad', 'Tallos'])
        parte['fecha'] = pd.to_datetime(parte['Fecha Corte'], errors='coerce')
        parte['anio'] = parte['anio_iso']
        parte['semana'] = parte['semana_iso']
        parte['k_finca'] = parte['Finca'].map(limpiar_finca)
        parte['k_producto'] = parte['Producto'].map(limpiar_nombre)
        parte['k_color'] = parte['Color'].map(limpiar_nombre)
        parte['k_variedad'] = parte['Variedad'].map(limpiar_nombre)
        parte['tallos_reales'] = pd.to_numeric(parte['Tallos'], errors='coerce')
        prod_detalle.append(parte)
    prod_detalle = pd.concat(prod_detalle, ignore_index=True)
    benchmark, control_cabana_union, control_excluidos, control_union = unir_benchmark(estimados, prod_detalle)
    controles_cabana = pd.concat([
        control_cabana_prod.assign(fuente='produccion_detalle'),
        control_cabana_union.assign(ano='todos'),
    ], ignore_index=True, sort=False)
    controles = control_prod.assign(tipo_control='produccion')
    controles = pd.concat([controles, control_est.assign(tipo_control='estimados')], ignore_index=True, sort=False)
    tablas = {
        'control_produccion': control_prod,
        'control_semanas_produccion': control_semanas,
        'control_cabana': controles_cabana,
        'control_estimados': control_est,
        'control_exclusiones_estimados': control_excluidos,
        'control_union': pd.DataFrame([control_union]),
        'produccion_semanal_limpia_multianual': produccion,
        'benchmark_consolidado_multianual': benchmark,
    }
    resultados = {
        'benchmark': benchmark,
        'tablas': tablas,
        'fecha_maxima_produccion': prod_detalle['fecha'].max(),
        'archivos_produccion_limpia': archivos_limpios,
    }
    exportar(resultados)
    print(f'Benchmark multi-anual: {len(benchmark):,} filas')
    print(f'Salida: {SALIDA}')
    return resultados




In [3]:
# Fase 1: importar y limpiar producción real 2021-2026
resultado_produccion = ejecutar_solo_produccion()

display(resultado_produccion['control_produccion'])
display(resultado_produccion['control_semanas'].head())
display(resultado_produccion['produccion'].head())

Produccion 2021: 926,798 filas limpias -> produccion_limpia_2021.parquet
Produccion 2022: 965,024 filas limpias -> produccion_limpia_2022.parquet
Produccion 2023: 1,137,303 filas limpias -> produccion_limpia_2023.parquet
Produccion 2024: 1,196,354 filas limpias -> produccion_limpia_2024.parquet
Produccion 2025: 1,223,917 filas limpias -> produccion_limpia_2025.parquet
Produccion 2026: 939,805 filas limpias -> produccion_limpia_2026.parquet
Produccion real limpia: 57,459 filas semanales agregadas
Salida: c:\Materias y Cosas Técnicas Uniandes\Proyecto de grado\Proyecto de grado 2\Datos\Producción real\datos_limpios_multianual


,filas_iniciales,filas_completamente_vacias,filas_finales,tipo_corte_imputados,tipo_corte_faltantes,fechas_invalidas,ano_libro,archivo_origen,hojas_leidas,errores_hojas,min_fecha,max_fecha,filas_llave_invalida_benchmark,salida_parquet
0,926798,0,926798,5455,547169,0,2021,Producción Acumulada 2021.xlsx,Sheet1,,2021-01-04,2022-01-02,0,c:\Materias y Cosas Técnicas Uniandes\Proyecto...
1,965024,0,965024,0,965024,0,2022,Producción Acumulada 2022.xlsx,Sheet1,,2022-01-03,2022-12-31,0,c:\Materias y Cosas Técnicas Uniandes\Proyecto...
2,1137303,0,1137303,23068,1020011,0,2023,Producción Acumulada 2023.xlsx,"Sheet1, Hoja1",,2023-01-02,2023-12-31,0,c:\Materias y Cosas Técnicas Uniandes\Proyecto...
3,1960155,763801,1196354,5572,0,0,2024,Producción Acumulada 2024.xlsx,"Sheet1, sem 45 en adelante",,2024-01-02,2024-12-28,0,c:\Materias y Cosas Técnicas Uniandes\Proyecto...
4,1223917,0,1223917,201,0,0,2025,Producción Acumulada 2025.xlsx,"01-44, 45-52",,2024-12-30,2025-12-27,0,c:\Materias y Cosas Técnicas Uniandes\Proyecto...
5,939835,30,939805,11610,0,0,2026,Producción Acumulada 2026.xlsx,BD,,2025-12-29,2026-09-05,0,c:\Materias y Cosas Técnicas Uniandes\Proyecto...


,anio,semana,dias_con_produccion,dias_faltantes_lunes_sabado,fechas_faltantes,estado_semana
0,2021,1,6,0,,completa
1,2021,2,6,0,,completa
2,2021,3,6,0,,completa
3,2021,4,6,0,,completa
4,2021,5,6,0,,completa


,anio,semana,k_finca,k_producto,k_color,k_variedad,tallos_reales
0,2021,1,ARABELLA,CARNATION,BICOLOR BURGUNDY,FREDY,5430
1,2021,1,ARABELLA,CARNATION,BICOLOR BURGUNDY,MERLETTO CRIMSON,4558
2,2021,1,ARABELLA,CARNATION,BICOLOR BURGUNDY,OLYMPIA,114
3,2021,1,ARABELLA,CARNATION,BICOLOR PINK,ANTIGUA,6243
4,2021,1,ARABELLA,CARNATION,BICOLOR PINK,ARMONIA,6243


In [4]:
# Control de la regla de respaldo: 2026-S02 debe usar el archivo de S03
inventario = construir_inventario()
seleccionados, archivos_duplicados, archivos_excluidos = seleccionar_archivos(inventario)
solicitudes_semanales = construir_solicitudes_semanales(seleccionados)

display(
    solicitudes_semanales.query(
        "ano_inicio == 2026 and semana_solicitada in [1, 2, 3]"
    )[[
        'ano_inicio', 'semana_solicitada', 'semana_inicio', 'archivo',
        'archivo_es_posterior', 'diferencia_semanas_archivo'
    ]]
)

,ano_inicio,semana_solicitada,semana_inicio,archivo,archivo_es_posterior,diferencia_semanas_archivo
156,2026,1,1,Estimado sem 1 a 8 de 2026 LG+AR envio.xlsx,False,0
157,2026,2,3,Estimado Sem 03-08 2026 Gaitana-Arabella.xlsx,True,1
158,2026,3,3,Estimado Sem 03-08 2026 Gaitana-Arabella.xlsx,False,0


In [5]:
# Fase 2 opcional: extraer estimados y generar benchmark multi-anual
# Ejecutar después de revisar la fase de producción.
def cargar_extraccion_estimados():
    resultados = ejecutar_extraccion()
    exportar_resultados(resultados)
    return resultados

resultado_completo = ejecutar()
display(resultado_completo['benchmark'].head())

Produccion 2021: 926,798 filas limpias -> produccion_limpia_2021.parquet
Produccion 2022: 965,024 filas limpias -> produccion_limpia_2022.parquet
Produccion 2023: 1,137,303 filas limpias -> produccion_limpia_2023.parquet
Produccion 2024: 1,196,354 filas limpias -> produccion_limpia_2024.parquet
Produccion 2025: 1,223,917 filas limpias -> produccion_limpia_2025.parquet
Produccion 2026: 939,805 filas limpias -> produccion_limpia_2026.parquet
Produccion real limpia: 57,459 filas semanales agregadas
Salida: c:\Materias y Cosas Técnicas Uniandes\Proyecto de grado\Proyecto de grado 2\Datos\Producción real\datos_limpios_multianual
Procesado archivo 1/178: 2023\Estimado Sem  01- 07  2023 Gaitana-Arabella.xlsx
Procesado archivo 2/178: 2023\Estimado Sem  02- 07  2023 Gaitana-Arabella.xlsx
Procesado archivo 3/178: 2023\Estimado Sem  03- 07  2023 Gaitana-Arabella.xlsx
Procesado archivo 4/178: 2023\Estimado Sem  05- 09  2023 Gaitana-Arabella.xlsx
Procesado archivo 5/178: 2023\Estimado Sem  06- 10  

FileNotFoundError: [Errno 2] No such file or directory: 'c:\\Materias y Cosas Técnicas Uniandes\\Proyecto de grado\\Proyecto de grado 2\\Datos\\Producción real\\datos_limpios_multianual\\produccion_limpia_por_anio\\produccion_limpia_2021.parquet'